# VT Research Direction

This notebook is the paper-run driver for the VT suite.

It explicitly uses `train / val / calib / test` splits, separates stopping depth from actual executed compute, and can sweep multiple backbones through the same fixed-$T$, VT, and attack pipeline.


In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

from vt_research_suite import (
    AttackConfig,
    DatasetConfig,
    EvaluationConfig,
    ModelConfig,
    TrainingConfig,
    apply_ablation,
    build_data_splits,
    build_model,
    build_stop_rule,
    collect_temporal_diagnostics,
    compare_stopping_rules,
    default_ablation_specs,
    evaluate_policy,
    fit_model,
    fixed_time_sweep,
    match_fixed_time_controls,
    rebuild_loader,
    seed_everything,
    select_compute_matched_threshold,
    select_threshold_neighbors,
    summarize_multiseed,
    threshold_sweep,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:
EXPERIMENT_ROOT = Path("vt_research_suite_outputs")
EXPERIMENT_ROOT.mkdir(exist_ok=True)

RUN_TRAINING = False
RUN_ABLATION_TRAINING = False
RUN_ARCHITECTURE_TRAINING = False
RUN_MULTI_SEED = False
RUN_EXTRA_DATASETS = False
PERSIST_OUTPUTS = True
REQUIRE_AUTOATTACK_FOR_FINAL = False
SHOW_NOTEBOOK_PROGRESS = True
RUN_FULL_TEST_THRESHOLD_SWEEP = True
RUN_MULTI_THRESHOLD_ATTACKS = True
RUN_AUTOATTACK_ON_ALL_SELECTED_THRESHOLDS = False
SELECTED_THRESHOLD_COUNT = 3
ATTACK_SUBSET_SAMPLES = 1000
ATTACK_SUBSET_BALANCED = True
ATTACK_SUBSET_SEED = 0

# Optional eval-loader overrides. Leave these as None to reuse dataset_cfg.batch_size.
# EXACT_VT_BATCH_SIZE only changes the staging batch size. When exact_vt_compute=True,
# VT evaluation still processes samples one-by-one inside each batch to keep compute exact.
CALIBRATION_BATCH_SIZE = None
TEST_BATCH_SIZE = None
ATTACK_BATCH_SIZE = None
EXACT_VT_BATCH_SIZE = None

# Keep exact VT compute on for main paper tables. Disable only for quick screening.
# Keep SynOps matching on for compute-matched VT vs fixed-T controls.
# For quick scouting, temporarily flip exact_vt_compute=False, then rerun shortlisted settings with exact_vt_compute=True.
dataset_cfg = DatasetConfig(
    name="cifar10",
    root="./data",
    batch_size=128,
    num_workers=2,
    seed=0,
    val_fraction=0.1,
    calibration_fraction=0.1,
    val_subset=None,
    calib_subset=None,
    test_subset=None,
)

model_cfg = ModelConfig(
    architecture="resnet",
    num_classes=10,
    in_channels=3,
    max_time_steps=8,
    stem_channels=128,
    stage_channels=(128, 256, 512),
    blocks_per_stage=(3, 3, 2),
    sigma_init_space="raw",
)

training_cfg = TrainingConfig(
    epochs=100,
    seed=0,
    device="cuda" if torch.cuda.is_available() else "cpu",
    output_dir=str(EXPERIMENT_ROOT / "checkpoints"),
    checkpoint_name="resnet_full_model.pt",
    mixed_time_steps=tuple(range(1, model_cfg.max_time_steps + 1)),
    selection_metric="clean_plus_robust",
)

eval_cfg = EvaluationConfig(
    vt_thresholds=(0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99),
    fixed_time_steps=tuple(range(1, model_cfg.max_time_steps + 1)),
    compute_metrics=True,
    exact_vt_compute=True,
    match_metric="avg_synops",
    max_batches=None,
)

selection_attack_cfg = AttackConfig(epsilon=8/255, alpha=2/255, steps=10, eot_iters=1)
adaptive_attack_cfg = AttackConfig(epsilon=8/255, alpha=2/255, steps=20, eot_iters=8, exit_aware=True)
compute_attack_cfg = replace(adaptive_attack_cfg, objective="compute_overuse", preserve_label_weight=1.0)
autoattack_cfg = AttackConfig(
    attack_name="autoattack",
    epsilon=8/255,
    exit_aware=adaptive_attack_cfg.exit_aware,
    force_full_trace=adaptive_attack_cfg.force_full_trace,
    attack_time_steps=model_cfg.max_time_steps,
    eot_iters=adaptive_attack_cfg.eot_iters,
)

print(dataset_cfg)
print(model_cfg)
print(training_cfg)
print(f"Main-table VT compute mode: {'sample_exact' if eval_cfg.exact_vt_compute else 'batched_proxy'}")
print(f"Main-table fixed-T match metric: {eval_cfg.match_metric}")
print(f"AutoAttack version policy: {autoattack_cfg.autoattack_version or ('rand' if autoattack_cfg.eot_iters > 1 else 'standard')}")
print(f"Selected-threshold panel size: {SELECTED_THRESHOLD_COUNT}")
print(
    f"Attack subset -> samples: {ATTACK_SUBSET_SAMPLES or 'full_test'}, "
    f"balanced: {ATTACK_SUBSET_BALANCED}, seed: {ATTACK_SUBSET_SEED}"
)
print(
    f"Eval loader overrides -> calib: {CALIBRATION_BATCH_SIZE or dataset_cfg.batch_size}, "
    f"test: {TEST_BATCH_SIZE or dataset_cfg.batch_size}, "
    f"attack: {ATTACK_BATCH_SIZE or TEST_BATCH_SIZE or dataset_cfg.batch_size}, "
    f"exact_vt: {EXACT_VT_BATCH_SIZE or CALIBRATION_BATCH_SIZE or TEST_BATCH_SIZE or dataset_cfg.batch_size}"
)


In [ ]:
seed_everything(training_cfg.seed)
splits = build_data_splits(dataset_cfg)

def load_or_train_model(current_model_cfg, current_training_cfg, current_splits):
    seed_everything(current_training_cfg.seed)
    model = build_model(current_model_cfg, seed=current_training_cfg.seed).to(current_training_cfg.device)
    checkpoint_path = Path(current_training_cfg.output_dir) / current_training_cfg.checkpoint_name
    history = None
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=current_training_cfg.device)
        model.load_state_dict(checkpoint["model_state_dict"])
        history = checkpoint.get("history")
        print(f"Loaded checkpoint: {checkpoint_path}")
    elif RUN_TRAINING:
        history, checkpoint_path = fit_model(
            model,
            current_splits.train_loader,
            current_splits.val_loader,
            current_training_cfg,
            selection_attack_config=selection_attack_cfg,
        )
        print(f"Trained and saved: {checkpoint_path}")
    else:
        print(f"Checkpoint not found: {checkpoint_path}")
        print("Set RUN_TRAINING = True to train before running the sweeps.")
    return model, history, checkpoint_path

model, history, checkpoint_path = load_or_train_model(model_cfg, training_cfg, splits)


In [ ]:
from importlib.util import find_spec

try:
    from tqdm.auto import tqdm as notebook_tqdm
except ImportError:
    notebook_tqdm = None


def progress_iter(iterable, desc, total=None, leave=False):
    if not SHOW_NOTEBOOK_PROGRESS or notebook_tqdm is None:
        return iterable
    resolved_total = total
    if resolved_total is None:
        try:
            resolved_total = len(iterable)
        except TypeError:
            resolved_total = None
    return notebook_tqdm(iterable, total=resolved_total, desc=desc, leave=leave, dynamic_ncols=True)


def persist_frame(frame, stem):
    if not PERSIST_OUTPUTS or frame is None:
        return None
    path = EXPERIMENT_ROOT / f"{stem}.csv"
    frame.to_csv(path, index=False)
    print(f"Saved {path}")
    return path


def require_autoattack_for_final_runs():
    if not REQUIRE_AUTOATTACK_FOR_FINAL:
        return
    if find_spec("torchattacks") is None and find_spec("autoattack") is None:
        raise RuntimeError(
            "AutoAttack is required for final runs but neither `torchattacks` nor `autoattack` is installed."
        )


def resolve_eval_loader(base_loader, batch_size_override=None):
    if batch_size_override is None or batch_size_override == base_loader.batch_size:
        return base_loader
    return rebuild_loader(base_loader, batch_size=batch_size_override, shuffle=False)


def get_calibration_loaders(current_splits):
    fixed_loader = resolve_eval_loader(current_splits.calib_loader, CALIBRATION_BATCH_SIZE)
    vt_batch_size = EXACT_VT_BATCH_SIZE if eval_cfg.exact_vt_compute else CALIBRATION_BATCH_SIZE
    vt_loader = resolve_eval_loader(current_splits.calib_loader, vt_batch_size)
    return {"fixed": fixed_loader, "vt": vt_loader}


def resolve_attack_loader(base_loader):
    attack_batch_size = ATTACK_BATCH_SIZE if ATTACK_BATCH_SIZE is not None else TEST_BATCH_SIZE
    attack_loader = resolve_eval_loader(base_loader, attack_batch_size)
    if ATTACK_SUBSET_SAMPLES is None:
        return attack_loader
    return build_subset_loader(
        attack_loader,
        ATTACK_SUBSET_SAMPLES,
        ATTACK_SUBSET_SEED,
        balanced=ATTACK_SUBSET_BALANCED,
        batch_size=attack_batch_size,
        shuffle=False,
    )


def get_test_loaders(current_splits):
    fixed_loader = resolve_eval_loader(current_splits.test_loader, TEST_BATCH_SIZE)
    vt_batch_size = EXACT_VT_BATCH_SIZE if eval_cfg.exact_vt_compute else TEST_BATCH_SIZE
    vt_loader = resolve_eval_loader(current_splits.test_loader, vt_batch_size)
    attack_loader = resolve_attack_loader(current_splits.test_loader)
    return {
        "fixed": fixed_loader,
        "vt": vt_loader,
        "attack": attack_loader,
        "diagnostics": attack_loader,
    }


def print_eval_loader_plan(current_splits):
    calibration_loaders = get_calibration_loaders(current_splits)
    test_loaders = get_test_loaders(current_splits)
    print(f"Calibration fixed/batched batch size: {calibration_loaders['fixed'].batch_size}")
    print(f"Calibration VT batch size: {calibration_loaders['vt'].batch_size}")
    print(f"Test fixed/batched batch size: {test_loaders['fixed'].batch_size}")
    print(f"Test VT batch size: {test_loaders['vt'].batch_size}")
    print(f"Attack/diagnostic batch size: {test_loaders['attack'].batch_size}")
    print(f"Attack/diagnostic samples: {len(test_loaders['attack'].dataset)}")
    if ATTACK_SUBSET_SAMPLES is not None:
        subset_mode = "class-balanced" if ATTACK_SUBSET_BALANCED else "random"
        print(f"Attack subset mode: {subset_mode} (seed={ATTACK_SUBSET_SEED})")
    if eval_cfg.exact_vt_compute:
        print(
            "Exact VT compute remains sample-exact inside each batch, so larger VT batch sizes help staging more than GPU saturation."
        )


def calibrate_compute_matched_threshold(current_model, current_splits, current_model_cfg, current_training_cfg, current_stop_rule, prefix, current_attack_config=None):
    calibration_loaders = get_calibration_loaders(current_splits)
    calibration = select_compute_matched_threshold(
        current_model,
        calibration_loaders["vt"],
        thresholds=eval_cfg.vt_thresholds,
        fixed_time_steps=eval_cfg.fixed_time_steps,
        stop_rule=current_stop_rule,
        time_steps=current_model_cfg.max_time_steps,
        attack_config=current_attack_config,
        compute_metrics=eval_cfg.compute_metrics,
        exact_vt_compute=eval_cfg.exact_vt_compute,
        match_metric=eval_cfg.match_metric,
        device=current_training_cfg.device,
        max_batches=eval_cfg.max_batches,
        vt_dataloader=calibration_loaders["vt"],
        fixed_dataloader=calibration_loaders["fixed"],
    )
    persist_frame(calibration["vt_results"], f"{prefix}_vt_threshold_sweep")
    persist_frame(calibration["fixed_results"], f"{prefix}_fixed_t_sweep")
    persist_frame(calibration["matches"], f"{prefix}_vt_fixed_matches")
    return calibration


print_eval_loader_plan(splits)


## 1. Calibration: Fixed-$T$ Controls and VT Thresholds

Calibration happens on the held-out calibration split, not the test split.

In [ ]:
vt_rule = build_stop_rule("max_softmax")
main_calibration = calibrate_compute_matched_threshold(model, splits, model_cfg, training_cfg, vt_rule, "main")
calib_vt = main_calibration["vt_results"]
calib_fixed = main_calibration["fixed_results"]
calib_matches = main_calibration["matches"]

display(calib_fixed[["time_steps", "accuracy", "avg_executed_steps", "avg_synops"]])
display(calib_vt[["threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "compute_mode"]])
display(calib_matches[["threshold", "vt_accuracy", "vt_avg_stopping_depth", "vt_avg_executed_steps", "matched_fixed_t", "fixed_accuracy", "synops_delta", "accuracy_delta"]].sort_values(["accuracy_delta", "synops_delta"], ascending=[False, True]))

selected_threshold = float(main_calibration["selected_threshold"])
selected_synops = float(main_calibration["selected_metric_value"])
selected_threshold_panel = select_threshold_neighbors(calib_vt, selected_threshold, count=SELECTED_THRESHOLD_COUNT)
selected_threshold_values = selected_threshold_panel["threshold"].tolist()
persist_frame(selected_threshold_panel, "main_selected_thresholds")

print(f"Selected threshold on calibration split: {selected_threshold:.2f}")
print(f"Matched fixed-T baseline from calibration: T={int(main_calibration['selected_match']['matched_fixed_t'])}")
print(f"Target SynOps for matched stopping-rule calibration: {selected_synops:.2f}")
display(selected_threshold_panel)


## 2. Fixed-$T$ Baseline then VT Test Evaluation


In [ ]:
test_loaders = get_test_loaders(splits)
selected_fixed_t = int(main_calibration["selected_match"]["matched_fixed_t"])

print("Running fixed-T baseline suite on test first...")
test_fixed_sweep = fixed_time_sweep(
    model,
    test_loaders["fixed"],
    time_step_values=eval_cfg.fixed_time_steps,
    compute_metrics=True,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
persist_frame(test_fixed_sweep, "test_fixed_t_sweep")

test_fixed_full = test_fixed_sweep.loc[test_fixed_sweep["time_steps"] == model_cfg.max_time_steps].iloc[0].to_dict()
test_fixed_full["report_mode"] = "fixed_full_horizon"

test_fixed_matched = test_fixed_sweep.loc[test_fixed_sweep["time_steps"] == selected_fixed_t].iloc[0].to_dict()
test_fixed_matched["report_mode"] = "fixed_compute_matched"

fixed_baseline_df = pd.DataFrame([test_fixed_full, test_fixed_matched])
persist_frame(fixed_baseline_df, "test_fixed_baseline_results")
display(fixed_baseline_df[["report_mode", "mode", "time_steps", "accuracy", "avg_executed_steps", "avg_synops", "avg_spike_count", "compute_mode"]])

print("Running VT suite on test for comparison...")
if RUN_FULL_TEST_THRESHOLD_SWEEP:
    test_vt_sweep = threshold_sweep(
        model,
        test_loaders["vt"],
        thresholds=eval_cfg.vt_thresholds,
        stop_rule=vt_rule,
        time_steps=model_cfg.max_time_steps,
        compute_metrics=True,
        exact_vt_compute=eval_cfg.exact_vt_compute,
        device=training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )
    test_vt_matches = match_fixed_time_controls(
        test_vt_sweep,
        test_fixed_sweep,
        match_metric=eval_cfg.match_metric,
    )

    persist_frame(test_vt_sweep, "test_vt_threshold_sweep")
    persist_frame(test_vt_matches, "test_vt_fixed_matches")

    selected_threshold_test_panel = selected_threshold_panel.merge(
        test_vt_sweep,
        on="threshold",
        how="left",
    )
else:
    selected_threshold_rows = []
    for _, threshold_row in selected_threshold_panel.iterrows():
        result = evaluate_policy(
            model,
            test_loaders["vt"],
            mode="vt",
            time_steps=model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=float(threshold_row["threshold"]),
            compute_metrics=True,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=training_cfg.device,
            max_batches=eval_cfg.max_batches,
        )
        result["threshold_role"] = threshold_row["threshold_role"]
        selected_threshold_rows.append(result)
    selected_threshold_test_panel = pd.DataFrame(selected_threshold_rows)

persist_frame(selected_threshold_test_panel, "test_selected_threshold_results")
test_vt = selected_threshold_test_panel.loc[selected_threshold_test_panel["threshold_role"] == "selected"].iloc[0].to_dict()
test_vt["report_mode"] = "vt_selected_threshold"

main_test_df = pd.DataFrame([test_fixed_full, test_fixed_matched, test_vt])
persist_frame(main_test_df, "main_test_results")
display(main_test_df[["report_mode", "mode", "time_steps", "threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "avg_spike_count", "compute_mode"]])
display(selected_threshold_test_panel[["threshold_role", "threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "compute_mode"]])

if RUN_FULL_TEST_THRESHOLD_SWEEP:
    display(test_vt_sweep[["threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "compute_mode"]])
    display(
        test_vt_matches[
            [
                "threshold",
                "vt_accuracy",
                "vt_avg_stopping_depth",
                "vt_avg_executed_steps",
                "matched_fixed_t",
                "fixed_accuracy",
                "synops_delta",
                "accuracy_delta",
            ]
        ].sort_values(["accuracy_delta", "synops_delta"], ascending=[False, True])
    )


## 3. Stopping-Rule Comparison Against the Fixed-$T$ Baseline


In [ ]:
calibration_loaders = get_calibration_loaders(splits)
test_loaders = get_test_loaders(splits)

rule_calibration = compare_stopping_rules(
    model,
    calibration_loaders["vt"],
    rule_names=eval_cfg.stopping_rules,
    candidate_thresholds=eval_cfg.vt_thresholds,
    time_steps=model_cfg.max_time_steps,
    target_value=selected_synops,
    compute_metrics=True,
    exact_vt_compute=eval_cfg.exact_vt_compute,
    target_metric=eval_cfg.match_metric,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
persist_frame(rule_calibration, "stopping_rule_calibration")
display(rule_calibration[["stop_rule", "threshold", "accuracy", "avg_synops", "metric_gap"]])

fixed_rule_anchor = test_fixed_full.copy()
fixed_rule_anchor["stop_rule"] = "fixed_T8_baseline"
fixed_rule_anchor["threshold"] = None

rule_test_rows = [fixed_rule_anchor]
for _, row in rule_calibration.iterrows():
    result = evaluate_policy(
        model,
        test_loaders["vt"],
        mode="vt",
        time_steps=model_cfg.max_time_steps,
        stop_rule=build_stop_rule(row["stop_rule"]),
        threshold=float(row["threshold"]),
        compute_metrics=True,
        exact_vt_compute=eval_cfg.exact_vt_compute,
        device=training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )
    result["stop_rule"] = row["stop_rule"]
    rule_test_rows.append(result)

rule_test_df = pd.DataFrame(rule_test_rows)
persist_frame(rule_test_df, "stopping_rule_test_results")
display(rule_test_df[["stop_rule", "threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops"]].sort_values(["stop_rule", "accuracy"], ascending=[True, False]))


## 4. Fixed-$T$ Baseline and VT Attacks


In [ ]:
attack_rows = []
require_autoattack_for_final_runs()
test_loaders = get_test_loaders(splits)
attack_threshold_panel = selected_threshold_panel if RUN_MULTI_THRESHOLD_ATTACKS else selected_threshold_panel[selected_threshold_panel["threshold_role"] == "selected"]
attack_objectives = ["untargeted", "wrong_early_exit", "compute_overuse"]

print("Running fixed-T baseline attacks first...")
fixed_attack = evaluate_policy(
    model,
    test_loaders["attack"],
    mode="fixed",
    time_steps=model_cfg.max_time_steps,
    attack_config=replace(selection_attack_cfg, steps=20, eot_iters=8),
    compute_metrics=True,
    exact_vt_compute=False,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
fixed_attack["threshold_role"] = "fixed_T8_baseline"
attack_rows.append(fixed_attack)

print("Running VT attacks against the selected threshold panel...")
for objective in progress_iter(attack_objectives, desc="Attack objectives", total=len(attack_objectives), leave=False):
    current_attack = replace(adaptive_attack_cfg if objective != "compute_overuse" else compute_attack_cfg, objective=objective)
    for _, threshold_row in attack_threshold_panel.iterrows():
        result = evaluate_policy(
            model,
            test_loaders["attack"],
            mode="vt",
            time_steps=model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=float(threshold_row["threshold"]),
            attack_config=current_attack,
            compute_metrics=True,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=training_cfg.device,
            max_batches=eval_cfg.max_batches,
        )
        result["threshold_role"] = threshold_row["threshold_role"]
        attack_rows.append(result)

try:
    autoattack_fixed = evaluate_policy(
        model,
        test_loaders["attack"],
        mode="fixed",
        time_steps=model_cfg.max_time_steps,
        attack_config=autoattack_cfg,
        compute_metrics=False,
        exact_vt_compute=False,
        device=training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )
    autoattack_fixed["threshold_role"] = "fixed_T8_baseline"
    attack_rows.append(autoattack_fixed)

    vt_autoattack_panel = attack_threshold_panel if RUN_AUTOATTACK_ON_ALL_SELECTED_THRESHOLDS else attack_threshold_panel[attack_threshold_panel["threshold_role"] == "selected"]
    for _, threshold_row in vt_autoattack_panel.iterrows():
        autoattack_vt = evaluate_policy(
            model,
            test_loaders["attack"],
            mode="vt",
            time_steps=model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=float(threshold_row["threshold"]),
            attack_config=autoattack_cfg,
            compute_metrics=False,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=training_cfg.device,
            max_batches=eval_cfg.max_batches,
        )
        autoattack_vt["threshold_role"] = threshold_row["threshold_role"]
        attack_rows.append(autoattack_vt)
except ImportError as exc:
    print(f"AutoAttack skipped: {exc}")

attack_df = pd.DataFrame(attack_rows)
persist_frame(attack_df, "attack_results")
display(attack_df[["mode", "threshold_role", "threshold", "attack_name", "autoattack_version", "attack_objective", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "attack_eot"]])

fixed_clean_diag = collect_temporal_diagnostics(
    model,
    test_loaders["diagnostics"],
    time_steps=model_cfg.max_time_steps,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
vt_clean_diag = collect_temporal_diagnostics(
    model,
    test_loaders["diagnostics"],
    time_steps=model_cfg.max_time_steps,
    stop_rule=vt_rule,
    threshold=selected_threshold,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
vt_attack_diag = collect_temporal_diagnostics(
    model,
    test_loaders["diagnostics"],
    time_steps=model_cfg.max_time_steps,
    stop_rule=vt_rule,
    threshold=selected_threshold,
    attack_config=compute_attack_cfg,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
steps = range(1, len(vt_clean_diag["avg_confidence_by_step"]) + 1)
axes[0].plot(steps, fixed_clean_diag["avg_confidence_by_step"], label="Fixed T=8")
axes[0].plot(steps, vt_clean_diag["avg_confidence_by_step"], label="VT Clean")
axes[0].plot(steps, vt_attack_diag["avg_confidence_by_step"], label="VT Compute Attack")
axes[0].set_title("Confidence Over Time")
axes[0].set_xlabel("Step")
axes[0].legend()
axes[1].plot(steps, fixed_clean_diag["avg_accuracy_by_step"], label="Fixed T=8")
axes[1].plot(steps, vt_clean_diag["avg_accuracy_by_step"], label="VT Clean")
axes[1].plot(steps, vt_attack_diag["avg_accuracy_by_step"], label="VT Compute Attack")
axes[1].set_title("Accuracy Over Time")
axes[1].set_xlabel("Step")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Fixed-T clean exit histogram:", fixed_clean_diag["step_histogram"])
print("VT clean exit histogram:", vt_clean_diag["step_histogram"])
print("VT compute-attack exit histogram:", vt_attack_diag["step_histogram"])


## 5. Minimal Ablation Matrix

Each variant is selected on validation, optionally calibrated on the calibration split, and only then reported on test.

In [ ]:
ablation_rows = []
ablation_dir = EXPERIMENT_ROOT / "ablations"
ablation_dir.mkdir(exist_ok=True)
ablation_specs = default_ablation_specs(model_cfg.max_time_steps)
test_loaders = get_test_loaders(splits)

for spec in progress_iter(ablation_specs, desc="Ablations", total=len(ablation_specs), leave=False):
    variant_model_cfg, variant_training_cfg = apply_ablation(model_cfg, training_cfg, spec)
    variant_training_cfg = replace(variant_training_cfg, output_dir=str(ablation_dir), checkpoint_name=f"{spec.name}.pt")
    seed_everything(variant_training_cfg.seed)
    variant_model = build_model(variant_model_cfg, seed=variant_training_cfg.seed).to(variant_training_cfg.device)
    variant_checkpoint = Path(variant_training_cfg.output_dir) / variant_training_cfg.checkpoint_name
    if variant_checkpoint.exists():
        checkpoint = torch.load(variant_checkpoint, map_location=variant_training_cfg.device)
        variant_model.load_state_dict(checkpoint["model_state_dict"])
    elif RUN_ABLATION_TRAINING:
        fit_model(variant_model, splits.train_loader, splits.val_loader, variant_training_cfg, selection_attack_config=selection_attack_cfg)
    else:
        print(f"Skipping {spec.name}; checkpoint not found.")
        continue

    fixed_eval = evaluate_policy(
        variant_model,
        test_loaders["fixed"],
        mode="fixed",
        time_steps=variant_model_cfg.max_time_steps,
        compute_metrics=True,
        exact_vt_compute=False,
        device=variant_training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )

    vt_eval = None
    chosen_threshold = None
    if spec.vt_enabled:
        ablation_calibration = calibrate_compute_matched_threshold(
            variant_model,
            splits,
            variant_model_cfg,
            variant_training_cfg,
            vt_rule,
            f"ablation_{spec.name}",
        )
        chosen_threshold = float(ablation_calibration["selected_threshold"])
        vt_eval = evaluate_policy(
            variant_model,
            test_loaders["vt"],
            mode="vt",
            time_steps=variant_model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=chosen_threshold,
            compute_metrics=True,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=variant_training_cfg.device,
            max_batches=eval_cfg.max_batches,
        )

    ablation_rows.append({
        "variant": spec.name,
        "selected_threshold": chosen_threshold,
        "fixed_accuracy": fixed_eval["accuracy"],
        "fixed_synops": fixed_eval["avg_synops"],
        "vt_accuracy": None if vt_eval is None else vt_eval["accuracy"],
        "vt_avg_executed_steps": None if vt_eval is None else vt_eval["avg_executed_steps"],
        "vt_synops": None if vt_eval is None else vt_eval["avg_synops"],
    })

ablation_df = pd.DataFrame(ablation_rows)
persist_frame(ablation_df, "ablation_results")
display(ablation_df)


## 6. Cross-Architecture Sweep

This uses the same train / val / calib / test pipeline for ResNet, VGG, and WRN.

In [ ]:
architecture_presets = {
    "resnet": replace(model_cfg, architecture="resnet", stem_channels=128, stage_channels=(128, 256, 512), blocks_per_stage=(3, 3, 2)),
    "vgg": replace(model_cfg, architecture="vgg", stem_channels=64, stage_channels=(64, 128, 256, 512, 512), blocks_per_stage=(1, 1, 2, 2, 2), dropout_rate=0.5),
    "wrn": replace(model_cfg, architecture="wrn", stem_channels=16, widen_factor=10, wrn_depth=28, stage_channels=(160, 320, 640), blocks_per_stage=(4, 4, 4)),
}

test_loaders = get_test_loaders(splits)
architecture_rows = []
architecture_items = list(architecture_presets.items())
for name, arch_model_cfg in progress_iter(architecture_items, desc="Architectures", total=len(architecture_items), leave=False):
    arch_training_cfg = replace(training_cfg, checkpoint_name=f"{name}_full_model.pt")
    seed_everything(arch_training_cfg.seed)
    arch_model = build_model(arch_model_cfg, seed=arch_training_cfg.seed).to(arch_training_cfg.device)
    arch_checkpoint = Path(arch_training_cfg.output_dir) / arch_training_cfg.checkpoint_name

    if arch_checkpoint.exists():
        checkpoint = torch.load(arch_checkpoint, map_location=arch_training_cfg.device)
        arch_model.load_state_dict(checkpoint["model_state_dict"])
    elif RUN_ARCHITECTURE_TRAINING:
        fit_model(arch_model, splits.train_loader, splits.val_loader, arch_training_cfg, selection_attack_config=selection_attack_cfg)
    else:
        print(f"Skipping {name}; checkpoint not found.")
        continue

    arch_calibration = calibrate_compute_matched_threshold(
        arch_model,
        splits,
        arch_model_cfg,
        arch_training_cfg,
        vt_rule,
        f"architecture_{name}",
    )
    arch_threshold = float(arch_calibration["selected_threshold"])

    arch_fixed = evaluate_policy(
        arch_model,
        test_loaders["fixed"],
        mode="fixed",
        time_steps=arch_model_cfg.max_time_steps,
        compute_metrics=True,
        exact_vt_compute=False,
        device=arch_training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )
    arch_vt = evaluate_policy(
        arch_model,
        test_loaders["vt"],
        mode="vt",
        time_steps=arch_model_cfg.max_time_steps,
        stop_rule=vt_rule,
        threshold=arch_threshold,
        compute_metrics=True,
        exact_vt_compute=eval_cfg.exact_vt_compute,
        device=arch_training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )

    architecture_rows.append({
        "architecture": name,
        "selected_threshold": arch_threshold,
        "fixed_accuracy": arch_fixed["accuracy"],
        "fixed_synops": arch_fixed["avg_synops"],
        "vt_accuracy": arch_vt["accuracy"],
        "vt_avg_executed_steps": arch_vt["avg_executed_steps"],
        "vt_synops": arch_vt["avg_synops"],
    })

architecture_df = pd.DataFrame(architecture_rows)
persist_frame(architecture_df, "architecture_results")
display(architecture_df)


## 7. Optional Multi-Seed Summary

This is intentionally left opt-in because the full paper run is expensive.

In [ ]:
if RUN_MULTI_SEED:
    multiseed_rows = []
    seed_values = list(eval_cfg.seeds)
    for seed in progress_iter(seed_values, desc="Seeds", total=len(seed_values), leave=False):
        seed_dataset_cfg = replace(dataset_cfg, seed=seed)
        seed_splits = build_data_splits(seed_dataset_cfg)
        seed_train_cfg = replace(training_cfg, seed=seed, checkpoint_name=f"{model_cfg.architecture}_seed{seed}.pt")
        seed_everything(seed)
        seed_model = build_model(model_cfg, seed=seed).to(seed_train_cfg.device)
        seed_checkpoint = Path(seed_train_cfg.output_dir) / seed_train_cfg.checkpoint_name
        if seed_checkpoint.exists():
            checkpoint = torch.load(seed_checkpoint, map_location=seed_train_cfg.device)
            seed_model.load_state_dict(checkpoint["model_state_dict"])
        else:
            fit_model(seed_model, seed_splits.train_loader, seed_splits.val_loader, seed_train_cfg, selection_attack_config=selection_attack_cfg)

        seed_calibration = calibrate_compute_matched_threshold(
            seed_model,
            seed_splits,
            model_cfg,
            seed_train_cfg,
            vt_rule,
            f"multiseed_seed{seed}",
        )
        seed_threshold = float(seed_calibration["selected_threshold"])
        seed_test_loaders = get_test_loaders(seed_splits)
        seed_eval = evaluate_policy(
            seed_model,
            seed_test_loaders["vt"],
            mode="vt",
            time_steps=model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=seed_threshold,
            compute_metrics=True,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=seed_train_cfg.device,
            max_batches=eval_cfg.max_batches,
        )
        seed_eval["selected_threshold"] = seed_threshold
        seed_eval["target_synops"] = float(seed_calibration["selected_metric_value"])
        seed_eval["seed"] = seed
        seed_eval["architecture"] = model_cfg.architecture
        multiseed_rows.append(seed_eval)

    multiseed_df = pd.DataFrame(multiseed_rows)
    multiseed_summary = summarize_multiseed(multiseed_rows, group_cols=["architecture"])
    persist_frame(multiseed_df, "multiseed_results")
    persist_frame(multiseed_summary, "multiseed_summary")
    display(multiseed_df)
    display(multiseed_summary)
else:
    print("Set RUN_MULTI_SEED = True to run the multi-seed summary.")
